# CycleGAN — Unpaired Image-to-Image Translation (2017)

_Papers · Generative Models_

**Translate between two image domains with NO matched pairs, by forcing a round-trip to return where it started.**

---

This notebook is a practice scaffold for the **CycleGAN — Unpaired Image-to-Image Translation (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn

torch.manual_seed(0)

# --- 0. Sanity-check the worked cycle-loss example (Eq. 2). ---
x   = torch.tensor([1.0, 0.0]);  FGx = torch.tensor([0.8, -0.1])   # x and F(G(x))
y   = torch.tensor([0.0, 1.0]);  GFy = torch.tensor([0.05, 0.9])   # y and G(F(y))
fwd = (FGx - x).abs().sum().item()      # |0.8-1.0|+|-0.1-0.0| = 0.30
bwd = (GFy - y).abs().sum().item()      # |0.05-0.0|+|0.9-1.0| = 0.15
print("worked example (Eq. 2):")
print("  forward  ||F(G(x))-x||_1 = %.2f" % fwd)          # 0.30
print("  backward ||G(F(y))-y||_1 = %.2f" % bwd)          # 0.15
print("  L_cyc = %.2f ;  lambda*L_cyc (lambda=10) = %.1f" % (fwd+bwd, 10*(fwd+bwd)))  # 0.45 ; 4.5


# --- 1. A toy UNPAIRED 2-domain task. ---
# Domain X: a blob near (-2, 0). Domain Y: the same blob ROTATED 90 deg + shifted.
# We draw X and Y INDEPENDENTLY -> the data is unpaired (no x is tied to any y).
R = torch.tensor([[0., -1.], [1., 0.]])                    # 90-degree rotation
def sample_X(m): return torch.randn(m, 2) * torch.tensor([0.9, 0.4]) + torch.tensor([-2.0, 0.0])
def sample_Y(m): return sample_X(m) @ R.T + torch.tensor([4.0, 0.0])   # independent draw, then rotate+shift


# --- 2. Generators G, F and discriminators D_X, D_Y, built by hand. ---
def gen(): return nn.Sequential(nn.Linear(2,64), nn.ReLU(), nn.Linear(64,64), nn.ReLU(), nn.Linear(64,2))
def dis(): return nn.Sequential(nn.Linear(2,64), nn.LeakyReLU(0.2), nn.Linear(64,64), nn.LeakyReLU(0.2), nn.Linear(64,1))


# --- 3. Train. use_cycle=False reproduces the ABLATION (adversarial loss only). ---
def train(use_cycle, steps=2500, lam=10.0):
    torch.manual_seed(0)
    G, F, Dx, Dy = gen(), gen(), dis(), dis()
    bce, l1 = nn.BCEWithLogitsLoss(), nn.L1Loss()
    optG = torch.optim.Adam(list(G.parameters()) + list(F.parameters()), lr=2e-3, betas=(0.5, 0.999))
    optD = torch.optim.Adam(list(Dx.parameters()) + list(Dy.parameters()), lr=2e-3, betas=(0.5, 0.999))
    m = 128
    for s in range(steps):
        x, y = sample_X(m), sample_Y(m)
        ones, zeros = torch.ones(m,1), torch.zeros(m,1)

        # (a) DISCRIMINATOR step: real vs fake for BOTH domains; detach the fakes.
        fy, fx = G(x).detach(), F(y).detach()
        lossD = (bce(Dy(y), ones) + bce(Dy(fy), zeros)
               + bce(Dx(x), ones) + bce(Dx(fx), zeros))
        optD.zero_grad(); lossD.backward(); optD.step()

        # (b) GENERATOR step: adversarial + (optionally) cycle-consistency (Eq. 2/3).
        adv = bce(Dy(G(x)), ones) + bce(Dx(F(y)), ones)    # fool both discriminators
        cyc = l1(F(G(x)), x) + l1(G(F(y)), y)              # round-trip L1 error
        lossG = adv + (lam * cyc if use_cycle else 0.0 * cyc)
        optG.zero_grad(); lossG.backward(); optG.step()

    # final round-trip error on fresh samples (the cycle-consistency we care about)
    with torch.no_grad():
        xt = sample_X(2000); fwd = (F(G(xt)) - xt).abs().sum(1).mean().item()
        yt = sample_Y(2000); bwd = (G(F(yt)) - yt).abs().sum(1).mean().item()
    return fwd, bwd

fwd_c, bwd_c = train(use_cycle=True)
fwd_n, bwd_n = train(use_cycle=False)
print("\nWITH    cycle loss: round-trip L1  X->Y->X = %.3f   Y->X->Y = %.3f" % (fwd_c, bwd_c))
print("WITHOUT cycle loss: round-trip L1  X->Y->X = %.3f   Y->X->Y = %.3f" % (fwd_n, bwd_n))
print("=> the cycle term drives the round trip home; without it the mapping is arbitrary.")
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does the cycle-consistency loss actually force the round trip F(G(x)) back to x — and what happens to that round-trip error if we ablate it?_

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)

# Toy UNPAIRED 2-domain task: X is a blob, Y is the same blob rotated 90 deg + shifted,
# drawn INDEPENDENTLY (no pairing). Track the round-trip error ||F(G(x))-x||_1 over training,
# with vs without the cycle-consistency loss (the ablation). Same loop as the notebook.
R = torch.tensor([[0., -1.], [1., 0.]])
def sample_X(m): return torch.randn(m, 2) * torch.tensor([0.9, 0.4]) + torch.tensor([-2.0, 0.0])
def sample_Y(m): return sample_X(m) @ R.T + torch.tensor([4.0, 0.0])
def gen(): return nn.Sequential(nn.Linear(2,64), nn.ReLU(), nn.Linear(64,64), nn.ReLU(), nn.Linear(64,2))
def dis(): return nn.Sequential(nn.Linear(2,64), nn.LeakyReLU(0.2), nn.Linear(64,64), nn.LeakyReLU(0.2), nn.Linear(64,1))

def run(use_cycle, steps=2500, lam=10.0):
    torch.manual_seed(0)
    G, F, Dx, Dy = gen(), gen(), dis(), dis()
    bce, l1 = nn.BCEWithLogitsLoss(), nn.L1Loss()
    oG = torch.optim.Adam(list(G.parameters())+list(F.parameters()), lr=2e-3, betas=(0.5,0.999))
    oD = torch.optim.Adam(list(Dx.parameters())+list(Dy.parameters()), lr=2e-3, betas=(0.5,0.999))
    m, hist = 128, []
    for s in range(steps):
        x, y = sample_X(m), sample_Y(m); o, z = torch.ones(m,1), torch.zeros(m,1)
        fy, fx = G(x).detach(), F(y).detach()
        lossD = bce(Dy(y),o)+bce(Dy(fy),z)+bce(Dx(x),o)+bce(Dx(fx),z)
        oD.zero_grad(); lossD.backward(); oD.step()
        adv = bce(Dy(G(x)),o)+bce(Dx(F(y)),o)
        cyc = l1(F(G(x)),x)+l1(G(F(y)),y)
        (adv + (lam*cyc if use_cycle else 0.0*cyc)).backward()
        oG.step(); oG.zero_grad()
        if s % 100 == 0: hist.append([s, round(cyc.item(), 3)])
    return hist

print("with cycle:   ", run(True))
print("without cycle:", run(False))
# With the cycle term the round-trip error -> ~0.07; without it it stays ~2.3.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The cycle-loss ablation (the paper's key experiment, on toy data). Train your CycleGAN twice:
            once with the full objective, once with the cycle term removed (adversarial loss only). Keep
            everything else identical and measure the average round-trip error $\|F(G(x))-x\|_1$ at the end.
            What happens to that error in each case, and what does it prove?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Run the full model; record the final round-trip $L_1$ error (it should fall close to 0). — _The cycle term explicitly minimizes this error, forcing $G$ and $F$ to be inverses._
- Set $\lambda = 0$ (drop $\mathcal{L}_{\text{cyc}}$ from the generator step only); retrain and measure the same round-trip error. — _An honest ablation changes exactly one thing &mdash; whether the round-trip constraint is enforced._
- Compare: with the cycle term the error is small; without it the error stays large because nothing ties $G(x)$ back to $x$. — _Adversarial loss only matches the output distribution; it permits any content-scrambling map._

**Answer:** With the full objective the round-trip error collapses toward $0$ (our toy run: ~0.07);
                 with the cycle term removed it stays large (~2.3) because the adversarial loss only asks
                 $G(x)$ to look like a real $Y$ sample &mdash; it never compares $x$ to its
                 reconstruction. This is exactly the paper's ablation result: cycle consistency is what makes
                 unpaired translation faithful rather than arbitrary.

</details>

**Problem 2.** Work a cycle loss by hand. For $x = [2.0,\ -1.0]$, suppose $G(x) = [-1.0,\ 2.0]$ and
            $F(G(x)) = [1.7,\ -0.6]$. Compute the forward round-trip $L_1$ error. If the backward round trip for
            some $y$ contributed $0.4$, what is $\mathcal{L}_{\text{cyc}}$ for this pair, and what is its
            weighted contribution with $\lambda = 10$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Forward error: $\|F(G(x))-x\|_1 = |1.7-2.0| + |-0.6-(-1.0)| = 0.3 + 0.4 = 0.7$. — _$L_1$ is the sum of absolute coordinate differences &mdash; note $G(x)$ itself never enters the error, only the reconstruction vs the original._
- Add the given backward term: $\mathcal{L}_{\text{cyc}} = 0.7 + 0.4 = 1.1$. — _Eq. 2 sums the forward and backward round-trip errors._
- Weight it: $\lambda\,\mathcal{L}_{\text{cyc}} = 10 \times 1.1 = 11.0$. — _Eq. 3 multiplies the cycle loss by $\lambda = 10$ before adding the adversarial losses._

**Answer:** Forward error $= 0.3 + 0.4 = 0.7$; total $\mathcal{L}_{\text{cyc}} = 0.7 + 0.4 = 1.1$;
                 weighted $10 \times 1.1 = 11.0$. Notice the intermediate $G(x)$ value is irrelevant to the
                 loss &mdash; only how well $F$ reconstructs the original $x$ matters.

</details>

**Problem 3.** A friend says: "If I only want horse&rarr;zebra ($G$), why bother training the reverse generator
            $F$? Just use $G$ with the adversarial loss." Explain what breaks, and why $F$ is essential even
            though you will throw it away at test time.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that adversarial loss on $G$ alone only matches the zebra distribution; it never references the input horse. — _Distribution-matching is invariant to permuting which input maps to which output._
- Observe you cannot even write a cycle loss without $F$ &mdash; there is no way to measure "did we get back to the original horse?". — _The round-trip $F(G(x))$ requires a map back from $Y$ to $X$._
- Conclude $F$ exists to make the cycle constraint computable; it anchors $G$ to preserve content. At test time you keep only $G$. — _$F$ is a training-time scaffold; the constraint it enables is what gives $G$ its faithfulness._

**Answer:** Without $F$ you cannot form the cycle loss at all, so $G$ is trained on adversarial loss
                 alone &mdash; which only forces its outputs to look like zebras, not to correspond to
                 the input horse. The mapping becomes arbitrary (any horse &rarr; any zebra). $F$ is the
                 training-time scaffold that makes $F(G(x))\approx x$ measurable, forcing $G$ to preserve the
                 horse's structure. You can discard $F$ after training, but you cannot train good unpaired $G$
                 without it.

</details>